#### Step1 - read the csv file using the spark dataframe reader API
(Note: needed to modify the bad data in time column in races.csv file)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
races_schema = StructType(fields=[StructField("raceId", IntegerType(), False),
                                StructField("year", IntegerType(), True),
                                StructField("round", IntegerType(), True),
                                StructField("circuitId", IntegerType(), False),
                                StructField("name", StringType(), True),
                                StructField("date", StringType(), True),
                                StructField("time", StringType(), True),
                                StructField("url", StringType(), True)])
df = spark.read.option("header", True).schema(races_schema).csv("/mnt/formula1dl4dbcourse/raw/races.csv")
#df.write.mode("overwrite").option("header", True).parquet("/mnt/formula1dl/processed/races

In [0]:
races_df=spark.read.option("header", True).schema(races_schema).csv("/mnt/formula1dl4dbcourse/raw/races.csv")
display(races_df)

##### step2 - add igestion date and race_timestamp to the dataframe

In [0]:
from pyspark.sql.functions import to_timestamp
def try_to_timestamp1(timestamp_string, pattern):
  try:
    return to_timestamp(lit(timestamp_string), pattern)
  except:
    print("returning none")
    return None

In [0]:
display(try_to_timestamp1("2021-03-28 08:00:00", "yyyy-MM-dd HH:mm:ss"))

In [0]:
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit
races_with_timestamp_df=races_df.withColumn("ingestion_date", current_timestamp())\
.withColumn("race_timestamp", to_timestamp(concat(col("date"), lit(" "), col("time")), "MM/dd/yyyy H:mm:ss"))



In [0]:
display(races_with_timestamp_df)

In [0]:
races_selected_df=races_with_timestamp_df.select(col("raceId").alias("race_id"), col("year").alias("race_year"), col("round"), col("circuitId").alias("circuit_id"), col("name"), col("ingestion_date"), col("race_timestamp"))
display(races_selected_df)


In [0]:
races_selected_df.write.mode("overwrite").format("parquet").saveAsTable("f1_processed.races")

In [0]:
display(spark.read.parquet("/mnt/formula1dl4dbcourse/processed/races"))

In [0]:
races_selected_df.write.mode("overwrite").partitionBy("race_year").format("parquet").saveAsTable("f1_processed.races") 

In [0]:
display(spark.read.parquet("/mnt/formula1dl4dbcourse/processed/races"))


In [0]:
dbutils.notebook.exit("Success")